# Prueba AB - Mejoras en la conversión

### 1. Cargar los datos. 

In [13]:
# cargar librerías

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

In [14]:
# cargar datasets

marketing_events = pd.read_csv('ab_project_marketing_events_us.csv')
ab_events = pd.read_csv('final_ab_events_upd_us.csv')
ab_new_users = pd.read_csv('final_ab_new_users_upd_us.csv')
ab_participants = pd.read_csv('final_ab_participants_upd_us.csv')

### 2. Visualización inicial de los datos

Se realiza la creación de una función para hacer la visualización inicial de los datos de una manera mucho más rapida.

In [15]:
def info_from_df(df):
    print(df.info())
    print("Cantidad de valores nulos por columna:")
    print(df.isnull().sum())
    print("Cantidad de filas duplicadas:")
    print(df.duplicated().sum())

In [22]:
df_list = [marketing_events, ab_events, ab_new_users, ab_participants]
for df in df_list:
    info_from_df(df)

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   name       14 non-null     str           
 1   regions    14 non-null     str           
 2   start_dt   14 non-null     datetime64[us]
 3   finish_dt  14 non-null     datetime64[us]
dtypes: datetime64[us](2), str(2)
memory usage: 580.0 bytes
None
Cantidad de valores nulos por columna:
name         0
regions      0
start_dt     0
finish_dt    0
dtype: int64
Cantidad de filas duplicadas:
0
<class 'pandas.DataFrame'>
RangeIndex: 423761 entries, 0 to 423760
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     423761 non-null  str           
 1   event_dt    423761 non-null  datetime64[us]
 2   event_name  423761 non-null  str           
 3   details     60314 non-null   float64       
dtypes: datetime64[us](1)

Es necesario hacer la conversión de datos para algunos dataframes con una función también. 

In [19]:
def change_date_format(df, column_name):
    for col in column_name:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    
    return df


In [21]:
marketing_events = change_date_format(marketing_events, ['start_dt','finish_dt'])
ab_events = change_date_format(ab_events, ['event_dt'])
ab_new_users = change_date_format(ab_new_users, ['first_date'])

### 3. Inicio de la conexión de los datos. 

In [26]:
final_participants_filtered = ab_participants[ab_participants['ab_test'] == 'recommender_system_test']

In [42]:
def get_conversion(df_eventos, df_pruebas, ab_test):
  if ab_test == None:
    result_df = df_eventos.merge(df_pruebas, on="user_id", how="inner")
  else:
    df_pruebas = df_pruebas[df_pruebas["ab_test"] == ab_test]
    result_df = df_eventos.merge(df_pruebas, on="user_id", how="inner")
  
  result_df = result_df[result_df["event_name"] != "login"]
  summary = pd.pivot_table(result_df, columns="event_name", index="group", aggfunc=pd.Series.nunique)["user_id"]
  summary = summary.reset_index()
  summary = summary[["group", "product_page", "product_cart", "purchase"]]
  summary["conversion_rate"] = summary["purchase"] / summary["product_page"]
  return summary

In [46]:
get_conversion(ab_events, ab_participants, None)

event_name,group,product_page,product_cart,purchase,conversion_rate
0,A,5208,2483,2682,0.514977
1,B,3986,2037,2008,0.503763


In [50]:
def get_conversion_by_event(summary):
    summary["conversion_rate_product_cart"] = summary["product_cart"] / summary["product_page"]
    summary["conversion_rate_purchase"] = summary["purchase"] / summary["product_cart"]
    return summary[['group','conversion_rate', 'conversion_rate_product_cart', 'conversion_rate_purchase']]

In [52]:
summary = get_conversion(ab_events, ab_participants, None)
total_conversions = get_conversion_by_event(summary)
total_conversions

event_name,group,conversion_rate,conversion_rate_product_cart,conversion_rate_purchase
0,A,0.514977,0.476767,1.080145
1,B,0.503763,0.511039,0.985763


In [62]:
def number_of_events(df_eventos, df_pruebas, ab_test):
  if ab_test == None:
    result_df = df_eventos.merge(df_pruebas, on="user_id", how="inner")
  else:
    df_pruebas = df_pruebas[df_pruebas["ab_test"] == ab_test]
    result_df = df_eventos.merge(df_pruebas, on="user_id", how="inner")
  
  number = result_df.groupby(["user_id", "group"])["event_name"].count().reset_index()
  A_count = number[number["group"] == "A"]["event_name"].mean()
  B_count = number[number["group"] == "B"]["event_name"].mean()
  return A_count, B_count

In [63]:
number_of_events(ab_events, ab_participants, None)

(np.float64(7.455549911099822), np.float64(7.11248992747784))